In [0]:
%sql
CREATE CATALOG IF NOT EXISTS my_first_catalog;

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS my_first_catalog.my_first_schema;

In [0]:
%sql
USE CATALOG my_first_catalog;
SHOW SCHEMAS;

In [0]:
from delta.tables import *
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, array, ArrayType, DateType, TimestampType, FloatType
from pyspark.sql.functions import *

In [0]:
df_sales_orders = spark.read                           \
                         .option("header", "true")      \
                         .option("inferSchema", "true") \
                         .csv("/Volumes/my_first_catalog/my_first_schema/my_staging_data/Data/store_orders_H1.csv")
df_sales_orders = df_sales_orders.withColumn("order_date", to_date(df_sales_orders.order_date, 'MM/dd/yyyy'))
display(df_sales_orders)
df_sales_orders.printSchema()

In [0]:
%sql
DROP TABLE IF EXISTS my_first_catalog.my_first_schema.store_orders;

In [0]:
df_sales_orders.write.format("delta").partitionBy("currency").saveAsTable("my_first_catalog.my_first_schema.store_orders")

In [0]:
%%sql		
SELECT * FROM my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.store_orders 
SET TBLPROPERTIES (
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
)

In [0]:
%%sql
DESCRIBE DETAIL my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
SHOW TBLPROPERTIES my_first_catalog.my_first_schema.store_orders;

In [0]:
%%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.store_orders WHERE order_number=5; 


In [0]:
%sql
UPDATE my_first_catalog.my_first_schema.store_orders SET sale_price=90.50 WHERE order_number=5;

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.store_orders WHERE order_number=5;

In [0]:
%%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.store_orders VERSION AS OF 0 WHERE order_number=5;

In [0]:
%sql
DELETE FROM my_first_catalog.my_first_schema.store_orders WHERE order_number=5;
SELECT * FROM my_first_catalog.my_first_schema.store_orders WHERE order_number=5;

In [0]:
%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
RESTORE TABLE my_first_catalog.my_first_schema.store_orders TO VERSION AS OF 2;
SELECT * FROM my_first_catalog.my_first_schema.store_orders VERSION AS OF 0 WHERE order_number=5; 

In [0]:
%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
SELECT count(*) FROM my_first_catalog.my_first_schema.store_orders;

In [0]:
df_store_orders_CDC1 = spark.read                          \
                         .option("header", "true")      \
                         .option("inferSchema", "true") \
                         .csv("/Volumes/my_first_catalog/my_first_schema/my_staging_data/Data/store_orders_H2.csv")
df_store_orders_CDC1 = df_store_orders_CDC1.withColumn("order_date", to_date(df_store_orders_CDC1.order_date, 'MM/dd/yyyy'))
display(df_store_orders_CDC1)
df_store_orders_CDC1.printSchema()
df_store_orders_CDC1.createOrReplaceTempView("cdc_store_orders_1")


In [0]:
spark.sql("SELECT * FROM cdc_store_orders").show()

In [0]:
%sql
MERGE INTO my_first_catalog.my_first_schema.store_orders
USING cdc_store_orders
ON cdc_store_orders.order_number = my_first_catalog.my_first_schema.store_orders.order_number
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *

In [0]:
%sql
SELECT count(*) FROM my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.store_orders;

In [0]:
df_store_orders_CDC2 = spark.read                          \
                         .option("header", "true")      \
                         .option("inferSchema", "true") \
                         .csv("/Volumes/my_first_catalog/my_first_schema/my_staging_data/Data/store_orders_H3.csv")

df_store_orders_CDC2 = df_store_orders_CDC2.withColumn("order_date",to_date(df_store_orders_CDC2.order_date, "MM/dd/yyyy"))

display(df_store_orders_CDC2)
df_store_orders_CDC2.printSchema()
df_store_orders_CDC2.createOrReplaceTempView("cdc_store_orders_2")

In [0]:
%sql
MERGE INTO my_first_catalog.my_first_schema.store_orders
USING cdc_store_orders_2
ON cdc_store_orders_2.order_number = my_first_catalog.my_first_schema.store_orders.order_number
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.store_orders WHERE order_number IN (500,1254, 1501, 2234, 2345);

In [0]:
%sql
SELECT count(*) FROM my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders;

In [0]:
df_sales_orders_CDC3 = spark.read                          \
                         .option("header", "true")      \
                         .option("inferSchema", "true") \
                         .csv("/Volumes/my_first_catalog/my_first_schema/my_staging_data/Data/store_orders_H4.csv")

df_sales_orders_CDC3 = df_sales_orders_CDC3.withColumn("order_date", to_date(df_sales_orders_CDC3.order_date, "MM/dd/yyyy"))

display(df_sales_orders_CDC3)
df_sales_orders_CDC3.printSchema()
df_sales_orders_CDC3.createOrReplaceTempView("cdc_store_orders_3")

In [0]:
%sql
MERGE INTO my_first_catalog.my_first_schema.store_orders
USING cdc_store_orders_3
ON cdc_store_orders_3.order_number = my_first_catalog.my_first_schema.store_orders.order_number
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.store_orders WHERE order_number IN (1501, 2345);

In [0]:
%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
DESCRIBE DETAIL my_first_catalog.my_first_schema.store_orders;


In [0]:
%sql
SELECT version, operation, isolationLevel
from (DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders);

In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.store_orders SET TBLPROPERTIES ('delta.isolationLevel' = 'Serializable')

In [0]:
%sql
UPDATE my_first_catalog.my_first_schema.store_orders SET sale_price=100.00 WHERE order_number=500;

In [0]:
%sql
SELECT version, operation, isolationLevel 
from (DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders);

In [0]:
%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
DESCRIBE DETAIL my_first_catalog.my_first_schema.store_orders

In [0]:
%sql
OPTIMIZE my_first_catalog.my_first_schema.store_orders
ZORDER BY (order_number)

In [0]:
%sql
DESCRIBE DETAIL my_first_catalog.my_first_schema.store_orders